In [1]:
import os
from pathlib import Path
import numpy as np
from tqdm.auto import tqdm
import pickle
from collections import defaultdict
import importlib
from matplotlib import pyplot as plt
from tqdm.auto import tqdm
from pathmapper import Project

from themachinethatgoesping.echosounders import kongsbergall, simradraw, index_functions
from themachinethatgoesping import pingprocessing as pp
from themachinethatgoesping import tools as ptools

In [2]:
import themachinethatgoesping as theping
theping.version()

themachinethatgoesping
- version:       0.33.0

modules:
- tools_nanopy   0.31.2
- tools          @PROJECT_VERSION@
- scripts        @PROJECT_VERSION@
- navigation_nanopy 0.18.2
- navigation     @PROJECT_VERSION@
- algorithms_nanopy 0.11.1
- algorithms     @PROJECT_VERSION@
- echosounders_nanopy 0.47.0
- echosounders   0.47.0
- pingprocessing_nanopy 0.13.0
- pingprocessing @PROJECT_VERSION@
- widgets        @PROJECT_VERSION@
- gridding       @PROJECT_VERSION@


In [3]:
#create the output path
path_out = "../unittest_data/kongsberg/kmall/"
os.makedirs(path_out,exist_ok=True)

project = Project('.')
project.update_paths(['path_in'])

{'path_in': '/home/users/data/SSPIRIT/data/sspirit_03/MB/Sis'}

## Explore and sort input data

In [4]:
path_in = str(project.paths.path_in)

folders = index_functions.find_folders_with_files(path_in, [".kmall",".kmwcd"], followlinks=True)
folders.sort()
N = 5

for input_path in folders:
    print(input_path)

Found 22 files
/home/users/data/SSPIRIT/data/sspirit_03/MB/Sis/OS01
/home/users/data/SSPIRIT/data/sspirit_03/MB/Sis/OS02
/home/users/data/SSPIRIT/data/sspirit_03/MB/Sis/OS03
/home/users/data/SSPIRIT/data/sspirit_03/MB/Sis/OS04
/home/users/data/SSPIRIT/data/sspirit_03/MB/Sis/eg


In [8]:
endings = [".kmall",".kmwcd"]
InputFileHandler = theping.echosounders.kmall.KMALLFileHandler


all_files = index_functions.find_files(folders, endings, followlinks=True, verbose=False)
prg = tqdm(total=len(all_files), desc="Creating test data", unit="files")

for input_path in folders:
    prg.set_postfix_str(f"{input_path}")

    prg.set_description(f"finding files")
    input_files = index_functions.find_files(input_path, endings, followlinks=True, verbose=False)

    if not input_files:
        print(f"- No {endings} files in {input_path}")
        continue

    for file in input_files:
        try:
            prg.set_postfix_str(f"{file}")

            # open and index files
            prg.set_description(f"opening files")

            fm = InputFileHandler(file, show_progress=False)
            
            if(len(fm.get_pings()) < N):
                raise RuntimeError(f"Less then {N} pings in file")

            # create new file name
            file_name, file_extension = os.path.splitext(os.path.split(file)[1])
            
            prg.set_description(f"creating new file name")
            output_file = os.path.join(
                input_path.replace(path_in, path_out), str(hash(file_name)) + file_extension
            )  # + hash of the old file name
            # print(output_file)
            os.makedirs(os.path.split(output_file)[0], exist_ok=True)

            prg.set_postfix_str(f"Writing file {output_file}")

            #get timestamps of all datagrams in Nth ping
            timestamps = [d.get_timestamp() for d in fm.get_pings()[N].file_data.datagrams()]
            max_timestamp_pings = np.nanmax(timestamps)

            # append timestamp of second PositionDatagram and second AttitudeDatagram
            for d in fm.datagram_interface.datagrams("#CPO"):
                if d.get_timestamp() > max_timestamp_pings:
                    timestamps.append(d.get_timestamp())
                    break
                    
            for d in fm.datagram_interface.datagrams("#SKM"):
                if d.get_timestamp() > max_timestamp_pings:
                    timestamps.append(d.get_timestamp())
                    break


            #compute largest timestamp
            max_timestamp = np.nanmax(timestamps)

            with open(output_file, "wb") as ofi:
                for d in fm.datagram_interface.datagrams():
                    if d.get_timestamp() > max_timestamp:
                        break
                    ofi.write(d.to_binary())
        except Exception as e:
            print(f"--- Error with {file} ---")
            print(f"Error: {e}")
            print("---")
        prg.update(1)


Creating test data:   0%|          | 0/22 [00:00<?, ?files/s]

In [9]:
files, index = theping.echosounders.index_functions.find_files_and_index(path_out,endings,index_root='/tmp/index/')

fh = InputFileHandler(files, index)

Found 22 files
indexing files ⢀ 88% :00s<00m:00s] [Found: 502 datagrams in 22 files (11MB)]                                         
Initializing ping interface ⢀ 90% :00s<00m:00s] [Done]                                              


In [10]:
# test pingview

pings = fh.get_pings()
wcpings = theping.pingprocessing.filter_pings.by_features(pings,['watercolumn.amplitudes'])
bpings = theping.pingprocessing.filter_pings.by_features(pings,['bottom.xyz'])

print(len(pings),len(wcpings),len(bpings))

99 58 81


In [14]:
%gui qt
wciviewer = theping.widgets.WCIViewerQt(wcpings)

In [12]:
echogram = theping.pingprocessing.watercolumn.echograms.EchogramBuilder.from_pings(wcpings)
echogram.set_y_axis_depth()

theping.widgets.EchogramViewerQt(echogram)

<themachinethatgoesping.widgets.echogramviewer_qt.EchogramViewerQt(0x585a32762270) at 0x788f0e8d3c80>

In [15]:
tiles = theping.pingprocessing.overview.map_builder.TileBuilder()
tiles.add_esri_ocean()

overview=theping.pingprocessing.overview.get_ping_overview(pings)

viewer = theping.widgets.MapViewerQt(tile_builder=tiles)
viewer.add_overview_track(overview)
viewer.connect_wci_viewer(wciviewer)